# CAPA GOLD 4 — MLflow Modeling y evaluación del modelo

**Proyecto:** Predicción de Retrasos en Entregas — Olist E-commerce  
**Responsable:** Marlon  
**Entrada:** `big_data_2026.olist.gold_ml_features`  
**Salida:** modelos registrados en MLflow + métricas + predicciones + importancia de variables

```text
03.1_gold_business_kpis
        ▼
03.2_gold_ml_features
        ▼
03.3_gold_dashboard_insights
        ▼
04_gold_mlflow_modeling       <-- estás aquí
        ├── entrenamiento Spark ML
        ├── tracking MLflow
        ├── evaluación temporal
        ├── feature importance
        └── tablas Gold de resultados
```

Este notebook es el paso de **modelado**. `03.2` ya hizo el contrato anti-fuga, imputación y split temporal. Aquí se hace el encoding, entrenamiento, evaluación, tracking MLflow y registro del campeón.

## 0. Configuración e imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array

import mlflow
import mlflow.spark
import pandas as pd
import numpy as np

CATALOG = "big_data_2026"
SCHEMA = "olist"
TABLA_ENTRADA = "gold_ml_features"
TABLA_METRICAS = "gold_ml_model_metrics"
TABLA_PREDICCIONES = "gold_ml_predictions"
TABLA_IMPORTANCIAS = "gold_ml_feature_importance"
NOMBRE_MODELO_UC = f"{CATALOG}.{SCHEMA}.olist_delay_predictor"
MLFLOW_EXPERIMENT = None
RANDOM_SEED = 42

def TBL(nombre: str) -> str:
    return f"{CATALOG}.{SCHEMA}.{nombre}"

print(f"Entrada : {TBL(TABLA_ENTRADA)}")
print(f"Modelo UC: {NOMBRE_MODELO_UC}")

## 1. Contrato de entrada

In [0]:
existentes = {fila.tableName for fila in spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()}
assert TABLA_ENTRADA in existentes, f"No existe {TBL(TABLA_ENTRADA)}. Ejecuta primero 03_2_gold_ml_features."

df = spark.table(TBL(TABLA_ENTRADA))
assert "is_late" in df.columns
assert "split_temporal" in df.columns
assert "order_id" in df.columns

n_total = df.count()
n_unicos = df.select("order_id").distinct().count()
assert n_total == n_unicos, f"La entrada no está a 1 fila por pedido: {n_total:,} vs {n_unicos:,}."
assert df.filter(~F.col("is_late").isin(0, 1)).count() == 0, "is_late contiene valores distintos de 0/1."

print(f"Pedidos: {n_total:,}")
display(df.groupBy("split_temporal", "is_late").count().orderBy("split_temporal", "is_late"))

## 2. Features y contrato anti-fuga

`03.2` ya excluyó las variables de fuga. Este notebook agrega una defensa adicional para detenerse si alguna reaparece.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    ByteType,
    ShortType,
    IntegerType,
    LongType,
    FloatType,
    DoubleType,
    DecimalType,
    BooleanType,
    DateType,
    TimestampType,
)

# ============================================================
# 1. COLUMNAS QUE NO SON FEATURES
# ============================================================

COLUMNAS_NO_FEATURE = {
    "order_id",
    "is_late",
    "split_temporal",
    "order_purchase_timestamp",
}


# ============================================================
# 2. DEFENSA ANTI-FUGA
# ============================================================

COLUMNAS_FUGA_DEFENSA = {
    "order_delivered_customer_date",
    "fecha_entrega_real",
    "days_delay",
    "dias_entrega_real",
    "severidad_retraso",
    "order_delivered_carrier_date",
    "fecha_envio_transportista",
    "dias_hasta_transportista",
    "order_status",
    "review_score",
    "fecha_review",
    "tiene_review",
}

fugas_presentes = COLUMNAS_FUGA_DEFENSA.intersection(df.columns)

assert not fugas_presentes, (
    f"Se detectaron columnas con potencial fuga: "
    f"{sorted(fugas_presentes)}"
)


# ============================================================
# 3. TRANSFORMAR AUTOMÁTICAMENTE LAS FECHAS
# ============================================================
#
# IMPORTANTE:
# Las fechas NO entran directamente al modelo.
#
# Para cada DateType/TimestampType seguro creamos:
#
#   *_year
#   *_month
#   *_day
#   *_dayofweek
#
# La fecha original se elimina de las features.
# ============================================================

TIPOS_FECHA = (DateType, TimestampType)

columnas_fecha_transformadas = []

for field in df.schema.fields:

    nombre = field.name

    # Ignorar columnas que ya sabemos que NO son features
    if nombre in COLUMNAS_NO_FEATURE:
        continue

    # Ignorar columnas de fuga
    if nombre in COLUMNAS_FUGA_DEFENSA:
        continue

    # Solo transformar fechas
    if isinstance(field.dataType, TIPOS_FECHA):

        print(f"Transformando fecha segura: {nombre}")

        df = (
            df
            .withColumn(
                f"{nombre}_year",
                F.year(F.col(nombre))
            )
            .withColumn(
                f"{nombre}_month",
                F.month(F.col(nombre))
            )
            .withColumn(
                f"{nombre}_day",
                F.dayofmonth(F.col(nombre))
            )
            .withColumn(
                f"{nombre}_dayofweek",
                F.dayofweek(F.col(nombre))
            )
        )

        # La fecha original no entra al modelo
        COLUMNAS_NO_FEATURE.add(nombre)

        columnas_fecha_transformadas.append(nombre)


print("\nFechas transformadas:")
for c in columnas_fecha_transformadas:
    print(f"  - {c}")


# ============================================================
# 4. IDENTIFICAR FEATURES
# ============================================================

feature_cols = [
    c
    for c in df.columns
    if c not in COLUMNAS_NO_FEATURE
    and c not in COLUMNAS_FUGA_DEFENSA
]


# ============================================================
# 5. SEPARAR NUMÉRICAS Y CATEGÓRICAS
# ============================================================

numeric_cols = []
categorical_cols = []

TIPOS_NUMERICOS = (
    ByteType,
    ShortType,
    IntegerType,
    LongType,
    FloatType,
    DoubleType,
    DecimalType,
    BooleanType,
)

for field in df.schema.fields:

    if field.name not in feature_cols:
        continue

    if isinstance(field.dataType, StringType):

        categorical_cols.append(field.name)

    elif isinstance(field.dataType, TIPOS_NUMERICOS):

        numeric_cols.append(field.name)

    elif isinstance(field.dataType, TIPOS_FECHA):

        # Si llega aquí significa que una fecha segura
        # no fue transformada correctamente.
        raise TypeError(
            f"La fecha {field.name} quedó sin transformar. "
            f"Tipo: {field.dataType}"
        )

    else:

        raise TypeError(
            f"Tipo no soportado como feature: "
            f"{field.name} -> {field.dataType}"
        )


# ============================================================
# 6. VALIDACIONES
# ============================================================

assert feature_cols, "No se encontraron features."

assert set(feature_cols) == set(
    numeric_cols + categorical_cols
), (
    "Hay features que no fueron clasificadas "
    "como numéricas o categóricas."
)


# ============================================================
# 7. RESUMEN
# ============================================================

print("\n" + "=" * 60)
print("CONTRATO DE FEATURES")
print("=" * 60)

print(f"Features totales : {len(feature_cols)}")
print(f"Numéricas        : {len(numeric_cols)}")
print(f"Categóricas      : {len(categorical_cols)}")
print(f"Fechas transformadas: {len(columnas_fecha_transformadas)}")

print("\nNuméricas:")
print(numeric_cols)

print("\nCategóricas:")
print(categorical_cols)

print("\nFechas originales excluidas:")
print(columnas_fecha_transformadas)

print("=" * 60)

## 3. Train/test temporal

In [0]:
# ============================================================
# SPLIT TEMPORAL TRAIN / TEST
# ============================================================

train = df.filter(
    F.col("split_temporal") == "train"
)

test = df.filter(
    F.col("split_temporal") == "test"
)

train_count = train.count()
test_count = test.count()

assert train_count > 0 and test_count > 0, (
    "Uno de los splits temporales quedó vacío."
)

# ============================================================
# VALIDACIÓN TEMPORAL
# ============================================================

max_train_date = (
    train
    .select(F.max("order_purchase_timestamp"))
    .first()[0]
)

min_test_date = (
    test
    .select(F.min("order_purchase_timestamp"))
    .first()[0]
)

assert max_train_date <= min_test_date, (
    "Violación temporal: train contiene pedidos "
    "posteriores al inicio de test."
)

# ============================================================
# RESUMEN
# ============================================================

print(f"Train: {train_count:,}")
print(f"Test : {test_count:,}")
print(f"Última fecha train: {max_train_date}")
print(f"Primera fecha test: {min_test_date}")

# ============================================================
# DISTRIBUCIÓN DEL TARGET — TRAIN
# ============================================================

display(
    train
    .groupBy("is_late")
    .count()
    .withColumn(
        "porcentaje",
        F.round(
            100 * F.col("count") / F.lit(train_count),
            2
        )
    )
    .orderBy("is_late")
)

# ============================================================
# DISTRIBUCIÓN DEL TARGET — TEST
# ============================================================

display(
    test
    .groupBy("is_late")
    .count()
    .withColumn(
        "porcentaje",
        F.round(
            100 * F.col("count") / F.lit(test_count),
            2
        )
    )
    .orderBy("is_late")
)

## 4. Preprocesamiento Spark ML

Las categóricas se transforman con `StringIndexer` + `OneHotEncoder`. El pipeline se ajusta únicamente sobre train cuando hacemos `fit(train)`. No se escala por defecto.

In [0]:
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}__idx", handleInvalid="keep") for c in categorical_cols]
encoded_cols = [f"{c}__ohe" for c in categorical_cols]

encoder = OneHotEncoder(
    inputCols=[f"{c}__idx" for c in categorical_cols],
    outputCols=encoded_cols,
    handleInvalid="keep"
) if categorical_cols else None

assembler_inputs = numeric_cols + encoded_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="keep")
preprocess_stages = indexers + ([encoder] if encoder is not None else []) + [assembler]

print(f"Indexers: {len(indexers)} | Encoder: {'sí' if encoder else 'no'} | entradas al vector: {len(assembler_inputs)}")

## 5. Funciones de evaluación

Se prioriza **PR-AUC**, además de ROC-AUC, F1, precision y recall. El threshold se selecciona usando únicamente train y después se congela para test.

In [0]:
def add_probability_column(predictions):
    return predictions.withColumn("prob_late", vector_to_array(F.col("probability"))[1])

def metricas_con_threshold(predictions, threshold):
    scored = (
        predictions
        .withColumn("prediction_threshold", (F.col("prob_late") >= F.lit(float(threshold))).cast("double"))
    )
    pr = BinaryClassificationEvaluator(labelCol="is_late", rawPredictionCol="probability", metricName="areaUnderPR")
    roc = BinaryClassificationEvaluator(labelCol="is_late", rawPredictionCol="probability", metricName="areaUnderROC")
    f1 = MulticlassClassificationEvaluator(labelCol="is_late", predictionCol="prediction_threshold", metricName="f1")
    precision = MulticlassClassificationEvaluator(labelCol="is_late", predictionCol="prediction_threshold", metricName="weightedPrecision")
    recall = MulticlassClassificationEvaluator(labelCol="is_late", predictionCol="prediction_threshold", metricName="weightedRecall")
    return {
        "pr_auc": float(pr.evaluate(scored)),
        "roc_auc": float(roc.evaluate(scored)),
        "f1": float(f1.evaluate(scored)),
        "precision": float(precision.evaluate(scored)),
        "recall": float(recall.evaluate(scored))
    }, scored

def seleccionar_threshold(train_predictions):
    candidatos = []
    for threshold in np.arange(0.10, 0.91, 0.05):
        metrics, _ = metricas_con_threshold(train_predictions, float(threshold))
        candidatos.append((float(threshold), metrics["f1"]))
    return max(candidatos, key=lambda x: x[1])

def confusion_binary(scored):
    return scored.groupBy("is_late", "prediction_threshold").count().orderBy("is_late", "prediction_threshold")

## 6. Entrenamiento y tracking MLflow

Se comparan dos modelos:

- **Logistic Regression** como baseline interpretable.
- **Random Forest** como modelo no lineal.

El campeón se selecciona por PR-AUC de test y, en empate, por F1.

In [0]:
import gc
import mlflow

def entrenar_modelo(nombre, estimator, params_log):

    pipeline = Pipeline(
        stages=preprocess_stages + [estimator]
    )

    with mlflow.start_run(run_name=nombre) as run:

        mlflow.log_params({
            "model_name": nombre,
            "target": "is_late",
            "split_strategy": "temporal_80_20_from_gold",
            "n_numeric_features": len(numeric_cols),
            "n_categorical_features": len(categorical_cols),
            "feature_count_before_encoding": len(feature_cols),
            "random_seed": RANDOM_SEED,
            **params_log
        })

        # ----------------------------------------------------
        # ENTRENAMIENTO
        # ----------------------------------------------------

        fitted = pipeline.fit(train)

        # ----------------------------------------------------
        # PREDICCIONES
        # ----------------------------------------------------

        train_pred = add_probability_column(
            fitted.transform(train)
        )

        test_pred = add_probability_column(
            fitted.transform(test)
        )

        # ----------------------------------------------------
        # THRESHOLD
        # ----------------------------------------------------

        threshold, _ = seleccionar_threshold(train_pred)

        train_metrics, train_scored = metricas_con_threshold(
            train_pred,
            threshold
        )

        test_metrics, test_scored = metricas_con_threshold(
            test_pred,
            threshold
        )

        # ----------------------------------------------------
        # MÉTRICAS MLFLOW
        # ----------------------------------------------------

        for k, v in train_metrics.items():
            mlflow.log_metric(f"train_{k}", float(v))

        for k, v in test_metrics.items():
            mlflow.log_metric(f"test_{k}", float(v))

        mlflow.log_metric(
            "decision_threshold",
            float(threshold)
        )

        # ----------------------------------------------------
        # GUARDAR MODELO EN MLFLOW
        # ----------------------------------------------------

        mlflow.spark.log_model(
            fitted,
            artifact_path="model"
        )

        # ----------------------------------------------------
        # INFORMACIÓN EN CONSOLA
        # ----------------------------------------------------

        print(
            f"\n{nombre} | Run: {run.info.run_id}"
        )

        print(
            f"Test PR-AUC: {test_metrics['pr_auc']:.4f} | "
            f"F1: {test_metrics['f1']:.4f} | "
            f"Recall: {test_metrics['recall']:.4f}"
        )

        # ----------------------------------------------------
        # COPIAR SOLO LOS RESULTADOS NECESARIOS
        # ----------------------------------------------------

        resultado = {
            "nombre": nombre,
            "run_id": run.info.run_id,
            "threshold": float(threshold),
            "train_metrics": train_metrics,
            "test_metrics": test_metrics,
        }

    # --------------------------------------------------------
    # LIBERAR REFERENCIAS PESADAS
    # --------------------------------------------------------

    del train_pred
    del test_pred
    del train_scored
    del test_scored
    del fitted
    del pipeline

    gc.collect()

    return resultado

## 7. Comparación y selección del campeón

In [0]:
comparacion = []
for r in resultados:
    m = r["test_metrics"]
    comparacion.append({
        "modelo": r["nombre"], "run_id": r["run_id"], "threshold": r["threshold"],
        "pr_auc": m["pr_auc"], "roc_auc": m["roc_auc"], "f1": m["f1"],
        "precision": m["precision"], "recall": m["recall"]
    })

pdf_comparacion = pd.DataFrame(comparacion).sort_values(["pr_auc", "f1"], ascending=False)
display(pdf_comparacion)

campeon = max(resultados, key=lambda r: (r["test_metrics"]["pr_auc"], r["test_metrics"]["f1"]))
print(f"CAMPEÓN: {campeon['nombre']} | PR-AUC={campeon['test_metrics']['pr_auc']:.4f} | F1={campeon['test_metrics']['f1']:.4f} | threshold={campeon['threshold']:.2f}")

## 8. Matriz de confusión del campeón

In [0]:
display(confusion_binary(campeon["test_scored"]))

## 9. Importancia de variables

Se obtiene `featureImportances` del Random Forest y se agregan las posiciones OHE para poder presentar importancia a nivel de feature original.

In [0]:
rf_result = next(r for r in resultados if r["nombre"] == "random_forest")
rf_stage = next(s for s in rf_result["pipeline"].stages if s.__class__.__name__ == "RandomForestClassificationModel")

def feature_names_from_fitted_pipeline(fitted_pipeline):
    names = list(numeric_cols)
    if categorical_cols:
        enc = next(s for s in fitted_pipeline.stages if s.__class__.__name__ == "OneHotEncoderModel")
        for i, c in enumerate(categorical_cols):
            size = int(enc.categorySizes[i])
            output_size = size - 1 if enc.getDropLast() and size > 0 else size
            names.extend([f"{c}__ohe_{j}" for j in range(output_size)])
    return names

feature_names = feature_names_from_fitted_pipeline(rf_result["pipeline"])
importances = rf_stage.featureImportances.toArray()
assert len(importances) == len(feature_names), f"Dimensiones incompatibles: {len(importances)} vs {len(feature_names)}"

pdf_importancia = pd.DataFrame({"feature_vector": feature_names, "importance": importances}).sort_values("importance", ascending=False)

def feature_original(nombre):
    for c in categorical_cols:
        if nombre.startswith(f"{c}__ohe_"):
            return c
    return nombre

pdf_importancia["feature_original"] = pdf_importancia["feature_vector"].map(feature_original)
pdf_importancia_agregada = pdf_importancia.groupby("feature_original", as_index=False)["importance"].sum().sort_values("importance", ascending=False)

display(pdf_importancia_agregada.head(25))

## 10. Registrar el campeón en MLflow / Unity Catalog

In [0]:
try:
    model_uri = f"runs:/{campeon['run_id']}/model"
    registered = mlflow.register_model(model_uri=model_uri, name=NOMBRE_MODELO_UC)
    print(f"Modelo registrado: {NOMBRE_MODELO_UC} | versión {registered.version}")
except Exception as e:
    print("AVISO: el modelo quedó guardado como artefacto del run MLflow, pero no se pudo registrar en Unity Catalog.")
    print(f"Detalle: {e}")

## 11. Guardar métricas en Gold

In [0]:
metric_rows = []
for r in resultados:
    for split_name, metrics in [("train", r["train_metrics"]), ("test", r["test_metrics"])]:
        for metric_name, value in metrics.items():
            metric_rows.append({
                "run_id": r["run_id"], "modelo": r["nombre"], "split": split_name,
                "metrica": metric_name, "valor": float(value),
                "decision_threshold": float(r["threshold"])
            })

metric_rows.append({
    "run_id": campeon["run_id"], "modelo": campeon["nombre"], "split": "test",
    "metrica": "campeon", "valor": 1.0, "decision_threshold": float(campeon["threshold"])
})

gold_metrics = spark.createDataFrame(metric_rows)
(gold_metrics.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(TBL(TABLA_METRICAS)))

spark.sql(f"COMMENT ON TABLE {TBL(TABLA_METRICAS)} IS 'Capa Gold. Métricas de modelos entrenados sobre gold_ml_features, con evaluación temporal y tracking MLflow. Responsable: Marlon.'")
display(spark.table(TBL(TABLA_METRICAS)))

## 12. Guardar predicciones del test

In [0]:
predicciones_test = (
    campeon["test_scored"]
    .select("order_id", "order_purchase_timestamp", "is_late", "prob_late", "prediction_threshold")
    .withColumn("modelo", F.lit(campeon["nombre"]))
    .withColumn("run_id", F.lit(campeon["run_id"]))
    .withColumn("decision_threshold", F.lit(float(campeon["threshold"])))
)

(predicciones_test.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(TBL(TABLA_PREDICCIONES)))
spark.sql(f"COMMENT ON TABLE {TBL(TABLA_PREDICCIONES)} IS 'Capa Gold. Predicciones del modelo campeón sobre el test temporal, con probabilidad de retraso y target observado. Responsable: Marlon.'")
display(spark.table(TBL(TABLA_PREDICCIONES)).orderBy(F.col("prob_late").desc()).limit(20))

## 13. Guardar importancia de variables

In [0]:
importance_rows = [
    {"run_id": campeon["run_id"], "modelo": "random_forest",
     "feature_original": row["feature_original"], "importance": float(row["importance"])}
    for _, row in pdf_importancia_agregada.iterrows()
]
gold_importance = spark.createDataFrame(importance_rows)
(gold_importance.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(TBL(TABLA_IMPORTANCIAS)))
spark.sql(f"COMMENT ON TABLE {TBL(TABLA_IMPORTANCIAS)} IS 'Capa Gold. Importancia de variables del Random Forest agregada por feature original. Responsable: Marlon.'")
display(spark.table(TBL(TABLA_IMPORTANCIAS)).orderBy(F.col("importance").desc()).limit(25))

## 14. Control final de calidad

In [0]:
assert spark.table(TBL(TABLA_METRICAS)).count() > 0
assert spark.table(TBL(TABLA_PREDICCIONES)).count() == test_count
assert spark.table(TBL(TABLA_IMPORTANCIAS)).count() > 0

m = campeon["test_metrics"]
print("="*70)
print("CAPA GOLD 4 — ENTRENAMIENTO FINALIZADO")
print("="*70)
print(f"Modelo campeón : {campeon['nombre']}")
print(f"MLflow Run ID  : {campeon['run_id']}")
print(f"PR-AUC test    : {m['pr_auc']:.4f}")
print(f"ROC-AUC test   : {m['roc_auc']:.4f}")
print(f"F1 test        : {m['f1']:.4f}")
print(f"Precision test : {m['precision']:.4f}")
print(f"Recall test    : {m['recall']:.4f}")
print(f"Threshold      : {campeon['threshold']:.2f}")
print(f"Métricas       : {TBL(TABLA_METRICAS)}")
print(f"Predicciones   : {TBL(TABLA_PREDICCIONES)}")
print(f"Importancias   : {TBL(TABLA_IMPORTANCIAS)}")
print(f"Modelo UC      : {NOMBRE_MODELO_UC}")